# 09 — Prédiction test et fichier de soumission

But : générer des prédictions sur `test_soundscapes`. Le format final dépend de la compétition/dataset ; ce notebook produit un CSV exploitable et un squelette de submission si `sample_submission.csv` existe.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


In [2]:
import pandas as pd
from src.data_loading import scan_soundscapes
from src.pipeline_steps import load_final_bundles
from src.soundscape import predict_soundscapes, labels_from_scores
from src.config import RESULT_DIR, SAMPLE_SUBMISSION_CSV, N_JOBS

test_files = scan_soundscapes(train=False)
print(test_files.shape)
bundles = load_final_bundles()
pred = predict_soundscapes(test_files, bundles, n_jobs=N_JOBS, k=5)
pred.to_csv(RESULT_DIR / 'test_soundscape_predictions_top3.csv', index=False)
display(pred.head())

(0, 0)


""


In [3]:
threshold_path = RESULT_DIR / 'threshold_tuning_soundscapes.csv'
BEST_THRESHOLD = 0.35
if threshold_path.exists():
    BEST_THRESHOLD = float(pd.read_csv(threshold_path).iloc[0]['threshold'])
print('Threshold utilisé:', BEST_THRESHOLD)
if not pred.empty:
    pred['pred_labels'] = [labels_from_scores(r.top_labels, r.top_scores, threshold=BEST_THRESHOLD) for r in pred.itertuples(index=False)]
    pred['birds'] = pred['pred_labels'].apply(lambda xs: ' '.join(xs) if xs else 'nocall')
    pred.to_csv(RESULT_DIR / 'test_soundscape_predictions_with_threshold.csv', index=False)
    display(pred[['filename','segment_id','birds']].head())

Threshold utilisé: 0.25


## Adapter au format `sample_submission`

Si le dataset impose une colonne `row_id`-> mapper `filename + segment_id` .

In [ ]:
if SAMPLE_SUBMISSION_CSV.exists():
    sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    print('Sample submission columns:', sub.columns.tolist())
    display(sub.head())
else:
    print('Pas de sample_submission.csv trouvé.')